In [1]:
!pip install python-dotenv

#mounting my google drive to store created dataset there
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/TinyStories/
!zip tinystories_de.zip /content/drive/MyDrive/tinystories_de.jsonl

  adding: content/drive/MyDrive/tinystories_de.jsonl (deflated 66%)


In [3]:
from google.colab import files
files.download("tinystories_de.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Translating Tiny Stories Data from English to German using NLLB (No Language Left Behind) Model

In [ ]:
# Authentication with Hugging face
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["HF_TOKEN"] = ""
login()



In [9]:
# Setting up the No Language Left behind model (600M paramenter) using Hugging face pipeline 
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

checkpoint = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to("cuda")


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [10]:
# Testing model on single text data 
text = "Once there was a small cat who loved milk."

inputs = tokenizer(text, return_tensors="pt").to("cuda")

translated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
    max_length=200
)

output = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

print(output)

['Es war einmal eine kleine Katze, die Milch liebte.']


In [7]:
# Loading the Tiny stories dataset
from datasets import load_dataset
dataset = load_dataset("roneneldan/TinyStories", split="train")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [11]:
# BATCH TRANSLATION USING MODEL 

def translate_batch(texts):

    tokenizer.src_lang = "eng_Latn"

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
        max_length=120
    )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [12]:
import os
print(os.getcwd())

/content


In [13]:
#Data set output path 
OUTPUT_FILE = "/content/drive/MyDrive/TinyStories/tinystories_de.jsonl"


In [18]:
# Resume From where convering is stopped 
import os

start_index = 0

if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        start_index = sum(1 for _ in f)

print("Resuming from:", start_index)

Resuming from: 1176


In [ ]:
# ENG -> DEUTSCH Conversion using NLLB

import json
from tqdm import tqdm

BATCH_SIZE = 8

with open(OUTPUT_FILE, "a", encoding="utf-8") as f:

    for i in tqdm(range(start_index, len(dataset), BATCH_SIZE)):

        batch_texts = dataset[i:i+BATCH_SIZE]["text"]

        german = translate_batch(batch_texts)

        for en, de in zip(batch_texts, german):

            json.dump(
                {"en": en, "de": de},
                f,
                ensure_ascii=False
            )
            f.write("\n")

        f.flush() 

  0%|          | 79/264818 [03:58<219:27:56,  2.98s/it]